# 🔍 Bloque 3 — Sesión 2: Datos incorrectos, codificación y cobertura

**Módulo:** Depuración, limpieza y Clasificación de Datos (5109)  
**Unidad Didáctica:** UD1 — Análisis exploratorio de datos  
**Duración estimada:** ~3 horas.

---

### Contenidos de esta sesión
- **3.3** Datos incorrectos: errores tipográficos, tipos mal asignados, formatos erróneos, duplicados
- **3.4** Mala codificación de variables: errores de encoding, mezcla de tipos en una columna
- **3.5** Evaluación de la cobertura: porcentaje de completitud por variable y por registro

### Criterio de evaluación asociado
> **(CE b)** Se han detectado datos incorrectamente introducidos, ausencia de valores, una mala codificación de las variables o valores atípicos.

In [ ]:
# Librerías necesarias
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import re

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 100

print('✅ Librerías cargadas correctamente')

---
## 3.3 Datos incorrectos

Más allá de los valores ausentes y los outliers, los datasets reales contienen frecuentemente datos incorrectos que no son `NaN` pero tampoco son válidos. Los tipos más habituales son:

| Tipo de error | Descripción | Ejemplo |
|---------------|-------------|----------|
| **Errores tipográficos** | Variantes ortográficas de un mismo valor | `'Madrid'`, `'madrid'`, `'MADRID'`, `'Madrd'` |
| **Tipos mal asignados** | El dato correcto, pero en el tipo incorrecto | Fecha almacenada como texto: `'2026-01-15'` |
| **Formatos erróneos** | Valores con formatos inconsistentes | Teléfonos: `'666123456'`, `'+34 666 12 34 56'`, `'666-12-34-56'` |
| **Duplicados** | Filas idénticas o casi idénticas repetidas | Mismo cliente registrado dos veces |
| **Valores imposibles** | Valores fuera del dominio válido | Edad = -5, porcentaje = 150, mes = 13 |

In [ ]:
# Dataset sintético con múltiples tipos de errores
datos_sucios = {
    'id':           [1, 2, 3, 4, 5, 6, 7, 8, 9, 2],            # id=2 duplicado
    'nombre':       ['Ana García', 'Luis Martínez', 'María López',
                     'carlos ruiz', 'ELENA PÉREZ', 'pedro  sanz',  # errores de capitalización/espacios
                     'Sofía Jiménez', 'Marcos Gil', 'Laura Torres', 'Luis Martínez'],
    'edad':         [23, 35, 28, -4, 31, 42, 150, 27, 33, 35],  # -4 y 150 imposibles
    'ciudad':       ['Madrid', 'madrid', 'Sevilla', 'Madrd',     # variantes de Madrid
                     'Barcelona', 'sevilla', 'Valencia',
                     'Barcelona', 'madrid', 'Madrid'],
    'telefono':     ['666123456', '+34 666 12 34 57', '666-12-34-58',
                     '666 12 34 59', '666123460', '666.12.34.61',
                     '6661234', '666123463', '66612346X', '666123457'],  # formatos y errores
    'fecha_alta':   ['2023-01-15', '15/02/2023', '2023-03-20',
                     '20-04-2023', '2023-05-10', '10 Jun 2023',  # formatos de fecha distintos
                     '2023-07-01', '2023-08-22', '2023-09-05', '2023-02-15'],
    'salario':      ['28.000 €', '35500', '32,500.00', '45000',
                     '38.000', 'N/A', '41000', '29500', '36000', '35500']
}

df = pd.DataFrame(datos_sucios)
print('Dataset con errores cargado:')
print(df.to_string())
print(f'\nDimensiones: {df.shape}')

### 📌 Ejemplo 1 — Detección y eliminación de duplicados

In [ ]:
# duplicated() devuelve True en las filas que son duplicado de una anterior
print('Filas duplicadas (completas):')
print(df[df.duplicated()])

# También podemos buscar duplicados por columnas específicas
print('\nDuplicados por columna "id":')
dup_id = df[df.duplicated(subset=['id'], keep=False)]
print(dup_id[['id', 'nombre', 'edad']])

# Eliminamos duplicados manteniendo la primera aparición
df_sin_dup = df.drop_duplicates(subset=['id'], keep='first')
print(f'\nFilas originales: {len(df)} | Después de deduplicar: {len(df_sin_dup)}')

### 📌 Ejemplo 2 — Detección de errores tipográficos y normalización de texto

In [ ]:
# Primero vemos los valores únicos de 'ciudad'
print('Valores únicos en "ciudad" (antes de limpiar):')
print(df['ciudad'].value_counts())

print('\n→ Madrid aparece como: "Madrid", "madrid", "Madrd"')
print('→ Sevilla aparece como: "Sevilla", "sevilla"')

In [ ]:
# Paso 1: normalizar capitalización y espacios
df['ciudad_norm'] = (
    df['ciudad']
    .str.strip()           # Eliminar espacios al inicio/fin
    .str.lower()           # Todo a minúsculas
    .str.title()           # Primera letra en mayúscula
)

print('Después de normalizar capitalización:')
print(df['ciudad_norm'].value_counts())
print('\n→ Quedan variantes distintas que son la misma ciudad:')
print('   "Madrd" (error tipográfico) debe corregirse manualmente')

In [ ]:
# Paso 2: corrección de errores tipográficos con diccionario de mapeo
# Para datasets pequeños, el mapeo manual es lo más fiable
mapa_ciudades = {
    'Madrd': 'Madrid',   # Error tipográfico conocido
}
df['ciudad_norm'] = df['ciudad_norm'].replace(mapa_ciudades)

print('Después de corregir errores tipográficos:')
print(df['ciudad_norm'].value_counts())

# Normalización del nombre: eliminar dobles espacios, strip
df['nombre_norm'] = (
    df['nombre']
    .str.strip()
    .str.replace(r'\s+', ' ', regex=True)  # Dobles espacios → uno
    .str.title()
)
print('\nNombres normalizados:')
print(df[['nombre', 'nombre_norm']])

### 📌 Ejemplo 3 — Detección de valores imposibles

In [ ]:
# Definimos rangos válidos para cada variable
reglas_dominio = {
    'edad': (0, 120),
}

print('Detección de valores fuera del dominio válido:')
for col, (minimo, maximo) in reglas_dominio.items():
    mascara = (df[col] < minimo) | (df[col] > maximo)
    problemas = df[mascara][['id', 'nombre', col]]
    print(f'\n  Columna "{col}" — rango válido: [{minimo}, {maximo}]')
    if problemas.empty:
        print('  ✅ Sin valores fuera de rango.')
    else:
        print(f'  ⚠️  {len(problemas)} valor/es fuera de rango:')
        print(problemas.to_string(index=False))

### 📌 Ejemplo 4 — Estandarización de formatos: teléfonos y fechas

In [ ]:
# Normalizamos los teléfonos: extraemos solo los 9 dígitos y validamos longitud
def normalizar_telefono(tel):
    """Extrae solo dígitos y valida que sean exactamente 9."""
    digitos = re.sub(r'\D', '', str(tel))  # Elimina todo lo que no sea dígito
    if digitos.startswith('34') and len(digitos) == 11:
        digitos = digitos[2:]              # Quita prefijo +34
    if len(digitos) == 9 and digitos.isdigit():
        return digitos
    return None  # Devuelve None si el formato no es válido

df['telefono_norm'] = df['telefono'].apply(normalizar_telefono)

print('Normalización de teléfonos:')
print(df[['telefono', 'telefono_norm']].to_string())

n_invalidos = df['telefono_norm'].isnull().sum()
print(f'\n⚠️  {n_invalidos} teléfono/s con formato inválido → convertidos a NaN')

In [ ]:
# Estandarización de fechas con formatos heterogéneos
print('Valores únicos en "fecha_alta":')
print(df['fecha_alta'].unique())

# pd.to_datetime con dayfirst=True e inferir el formato
df['fecha_norm'] = pd.to_datetime(df['fecha_alta'], dayfirst=False, errors='coerce', infer_datetime_format=True)

# Los formatos que no se pueden parsear automáticamente quedan como NaT
print('\nFechas normalizadas:')
print(df[['fecha_alta', 'fecha_norm']].to_string())

n_nat = df['fecha_norm'].isnull().sum()
if n_nat > 0:
    print(f'\n⚠️  {n_nat} fecha/s no parseada/s automáticamente → revisar manualmente')

---
## 3.4 Mala codificación de variables

La codificación de variables se refiere a cómo se representan internamente los datos. Los problemas más frecuentes son:

| Problema | Descripción | Ejemplo |
|----------|-------------|----------|
| **Encoding de texto incorrecto** | El archivo se lee con una codificación distinta a la que se usó para escribirlo | `'caf\xe9'` en lugar de `'café'` |
| **Mezcla de tipos en una columna** | Una columna contiene tanto números como texto | `[12, '15', 'N/A', 18.5]` |
| **Booleano codificado como texto** | `'Sí'/'No'`, `'True'/'False'`, `'1'/'0'` como cadenas | `df['activo'] = ['Sí', 'No', 'Sí']` |
| **Categórica ordinal sin orden definido** | La variable tiene orden natural pero pandas no lo conoce | `['Bajo', 'Alto', 'Medio']` sin `CategoricalDtype` |
| **Numérica almacenada como texto** | Números con separadores o símbolos que impiden operar con ellos | `'28.000 €'`, `'32,500.00'` |

### 📌 Ejemplo 5 — Mezcla de tipos en una columna y coerción

In [ ]:
# La columna 'salario' tiene múltiples problemas: símbolos, puntos, comas, texto
print('Columna "salario" original:')
print(df[['id', 'salario']])
print(f'\nTipo detectado por pandas: {df["salario"].dtype}')

In [ ]:
def limpiar_numero(valor):
    """Convierte una cadena con formato numérico heterogéneo a float."""
    s = str(valor).strip()
    s = s.replace('€', '').strip()    # Elimina símbolo de moneda
    s = s.replace('.', '')            # Elimina separador de miles (formato ES)
    s = s.replace(',', '.')           # Convierte separador decimal ES → EN
    try:
        return float(s)
    except ValueError:
        return np.nan                  # 'N/A' u otros no numéricos → NaN

df['salario_num'] = df['salario'].apply(limpiar_numero)

print('Salario limpio:')
print(df[['id', 'salario', 'salario_num']].to_string())
print(f'\nTipo resultante: {df["salario_num"].dtype}')
print(f'Valores ausentes tras limpieza: {df["salario_num"].isnull().sum()}')

### 📌 Ejemplo 6 — Problemas de encoding de texto (charset)

In [ ]:
# Simulamos un CSV con encoding latin-1 guardado con nombre UTF-8
# Situación típica: archivos generados por Excel en España
csv_latin1 = 'nombre,ciudad,profesión\n'
csv_latin1 += 'José,Málaga,Médico\n'
csv_latin1 += 'María,Cádiz,Física\n'

# Guardamos en latin-1
with open('test_latin1.csv', 'w', encoding='latin-1') as f:
    f.write(csv_latin1)

# Intentamos leer como UTF-8 → error de encoding
print('❌ Leyendo con encoding=utf-8 (incorrecto):')
try:
    df_error = pd.read_csv('test_latin1.csv', encoding='utf-8')
    print(df_error)
except UnicodeDecodeError as e:
    print(f'   UnicodeDecodeError: {e}')

# Leemos con el encoding correcto
print('\n✅ Leyendo con encoding=latin-1 (correcto):')
df_correcto = pd.read_csv('test_latin1.csv', encoding='latin-1')
print(df_correcto)

print('\n💡 Consejo: si no sabes el encoding, prueba "utf-8" primero,')
print('   luego "latin-1" (ISO-8859-1) o "cp1252" (Windows).')
print('   La librería "chardet" puede detectarlo automáticamente:')
print('   >>> import chardet; chardet.detect(open("archivo.csv", "rb").read())')

### 📌 Ejemplo 7 — Codificación correcta de variables categóricas ordinales

In [ ]:
# Dataset con variables ordinales mal codificadas
encuesta = pd.DataFrame({
    'id':            [1, 2, 3, 4, 5, 6],
    'satisfaccion':  ['Alto', 'Bajo', 'Medio', 'Alto', 'Bajo', 'Medio'],
    'nivel_estudio': ['Universitario', 'Primaria', 'Secundaria',
                      'Universitario', 'Primaria', 'Secundaria'],
    'puntuacion':    [8, 3, 5, 9, 2, 6]
})

print('Dataset original — satisfaccion como object:')
print(encuesta.dtypes)
print()
print(encuesta)

# Problema: si ordenamos por satisfacción, pandas usa orden alfabético
print('\n❌ Orden incorrecto (alfabético):')
print(encuesta.sort_values('satisfaccion')['satisfaccion'].values)

In [ ]:
# Solución: CategoricalDtype con el orden correcto
from pandas.api.types import CategoricalDtype

orden_satisfaccion = CategoricalDtype(
    categories=['Bajo', 'Medio', 'Alto'],
    ordered=True
)
orden_estudios = CategoricalDtype(
    categories=['Primaria', 'Secundaria', 'Universitario'],
    ordered=True
)

encuesta['satisfaccion']  = encuesta['satisfaccion'].astype(orden_satisfaccion)
encuesta['nivel_estudio'] = encuesta['nivel_estudio'].astype(orden_estudios)

print('✅ Tipos corregidos:')
print(encuesta.dtypes)

print('\n✅ Orden correcto tras ordenar por satisfaccion:')
print(encuesta.sort_values('satisfaccion')[['id', 'satisfaccion', 'puntuacion']].to_string())

print('\n✅ Comparaciones ordinales ahora funcionan:')
print(encuesta['satisfaccion'] > 'Bajo')

### 📌 Ejemplo 8 — Detección de mezcla de tipos en una columna

In [ ]:
# Una columna con tipo 'object' puede contener datos de tipos distintos
columna_mezclada = pd.Series([12, '15', None, 18.5, 'veinte', True, '21'])

print('Columna con tipos mezclados:')
for i, val in enumerate(columna_mezclada):
    print(f'  [{i}] valor={repr(val):15s} tipo={type(val).__name__}')

# Detectamos qué tipos hay en la columna
tipos_presentes = columna_mezclada.dropna().apply(type).value_counts()
print('\nTipos encontrados en la columna:')
print(tipos_presentes)

# Intentamos convertir a numérico con coerción
col_numerica = pd.to_numeric(columna_mezclada, errors='coerce')
print('\nTras pd.to_numeric(errors="coerce"):')
print(col_numerica)
print('⚠️  "veinte" y True no convertibles → NaN')

---
### 🔧 Ejercicio 1 — Detección y corrección de errores en un dataset de ventas

Analiza el siguiente dataset de ventas y corrige todos los errores que encuentres.

In [ ]:
ventas = pd.DataFrame({
    'id_pedido':   [1001, 1002, 1003, 1004, 1005, 1006, 1007, 1002],
    'cliente':     ['ACME Corp', 'Beta SL', 'acme corp', 'Gamma SA',
                    'BETA SL', 'Delta Inc', 'acme corp', 'Beta SL'],
    'producto':    ['Teclado', 'Monitor', 'Raton', 'Teclado',
                    'Monitor', 'Hub USB', 'Ratón', 'Monitor'],
    'unidades':    [3, 1, 'dos', 5, 2, -1, 4, 1],        # 'dos' y -1 son errores
    'precio_unit': ['89,99 €', '349.00', '45.50 €', '89.99',
                    '349,00 €', '29,99 €', '45.50', '349.00'],
    'fecha':       ['2026-01-10', '10/01/2026', '2026-01-12',
                    '12-01-2026', '2026-01-15', '15 Jan 2026',
                    '2026-01-18', '2026-01-10'],
    'enviado':     ['Sí', 'No', 'SI', 'sí', 'no', 'No', 'Sí', 'No']
})

print('Dataset de ventas con errores:')
print(ventas.to_string())

In [ ]:
ventas_limpio = ventas.copy()

# TAREA 1: Detecta y elimina filas duplicadas (por id_pedido)
# ventas_limpio = ventas_limpio.drop_duplicates(subset=['id_pedido'], keep='first')

# TAREA 2: Normaliza la columna 'cliente' (elimina variantes del mismo nombre)
# ventas_limpio['cliente'] = ventas_limpio['cliente'].str.strip().str.title()

# TAREA 3: Normaliza 'producto' (hay dos variantes de 'Ratón'/'Raton')
# ...

# TAREA 4: Convierte 'unidades' a entero.
#   - 'dos' debe convertirse a NaN (pd.to_numeric con errors='coerce')
#   - Los valores negativos son imposibles → convertirlos a NaN
# ...

# TAREA 5: Limpia 'precio_unit' y conviértelo a float
#   (pista: usa la función limpiar_numero definida antes, adaptándola)
# ...

# TAREA 6: Normaliza 'fecha' a datetime con pd.to_datetime
# ...

# TAREA 7: Convierte 'enviado' a booleano (True/False)
#   Pista: normaliza primero a minúsculas, luego mapea
# ...

# TAREA 8: Muestra el DataFrame limpio con sus tipos correctos
# print(ventas_limpio.dtypes)
# print(ventas_limpio)

print('Completa las tareas comentadas arriba.')

---
## 3.5 Evaluación de la cobertura

La **cobertura** (o completitud) mide qué proporción de los datos esperados están realmente disponibles. Es una dimensión clave de la **calidad del dato**.

Se puede calcular a dos niveles:

```
Cobertura por variable  = (nº de valores presentes) / (nº total de registros)
Cobertura por registro  = (nº de campos rellenos en una fila) / (nº total de columnas)
Cobertura global        = (nº total de celdas no nulas) / (nº total de celdas)
```

> 💡 En proyectos reales es habitual definir **umbrales de aceptabilidad**:
> - Variables con cobertura < 50% → candidatas a ser descartadas del modelo
> - Registros con cobertura < 50% → candidatos a ser eliminados del entrenamiento

### 📌 Ejemplo 9 — Cobertura por variable y por registro

In [ ]:
titanic = sns.load_dataset('titanic')

# --- Cobertura por variable ---
cobertura_col = (1 - titanic.isnull().mean()) * 100
cobertura_col = cobertura_col.sort_values()

print('Cobertura (completitud) por variable — Titanic:')
for col, pct in cobertura_col.items():
    barra_ok  = '█' * int(pct / 5)
    barra_fal = '░' * (20 - int(pct / 5))
    estado = '✅' if pct == 100 else ('⚠️ ' if pct >= 80 else '❌')
    print(f'  {estado} {col:20s}: {pct:5.1f}% |{barra_ok}{barra_fal}|')

In [ ]:
# --- Cobertura por registro ---
cobertura_fila = (1 - titanic.isnull().mean(axis=1)) * 100

print('Distribución de cobertura por registro:')
bins = [0, 50, 70, 80, 90, 100]
etiquetas = ['<50%', '50-70%', '70-80%', '80-90%', '90-100%']
grupos = pd.cut(cobertura_fila, bins=bins, labels=etiquetas, right=True)
print(grupos.value_counts().sort_index())

print(f'\nRegistros con cobertura < 80%: {(cobertura_fila < 80).sum()} ({(cobertura_fila < 80).mean()*100:.1f}%)')

# --- Cobertura global ---
n_total  = titanic.shape[0] * titanic.shape[1]
n_presentes = titanic.count().sum()
cobertura_global = n_presentes / n_total * 100
print(f'\nCobertura global del dataset: {cobertura_global:.1f}%')

### 📌 Ejemplo 10 — Visualización de la cobertura

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico de barras: cobertura por variable
colores_col = ['#C44E52' if v < 80 else ('#F0B429' if v < 95 else '#55A868')
               for v in cobertura_col.values]
bars = axes[0].barh(cobertura_col.index, cobertura_col.values,
                    color=colores_col, edgecolor='white')
axes[0].axvline(x=80, color='orange', linestyle='--', alpha=0.7, label='Umbral 80%')
axes[0].axvline(x=95, color='green',  linestyle='--', alpha=0.7, label='Umbral 95%')
axes[0].set_xlim(0, 110)
axes[0].set_xlabel('% de completitud')
axes[0].set_title('Cobertura por variable')
axes[0].legend(fontsize=8)
for bar, val in zip(bars, cobertura_col.values):
    axes[0].text(val + 0.5, bar.get_y() + bar.get_height()/2,
                 f'{val:.1f}%', va='center', fontsize=7)

# Histograma: distribución de cobertura por registro
axes[1].hist(cobertura_fila, bins=20, color='steelblue', edgecolor='white')
axes[1].axvline(x=80, color='orange', linestyle='--', label='Umbral 80%')
axes[1].set_xlabel('% de completitud del registro')
axes[1].set_ylabel('Nº de registros')
axes[1].set_title('Distribución de cobertura por registro')
axes[1].legend()

plt.tight_layout()
plt.show()

### 📌 Ejemplo 11 — Informe de cobertura completo y exportación

In [ ]:
def informe_cobertura(df, umbral_variable=80.0, umbral_registro=50.0):
    """
    Genera un informe de cobertura completo y devuelve un DataFrame resumen.
    
    Parámetros:
    -----------
    umbral_variable : % mínimo de completitud aceptable por columna
    umbral_registro : % mínimo de completitud aceptable por fila
    """
    n_filas, n_cols = df.shape
    n_total = n_filas * n_cols

    # Cobertura por variable
    ausentes   = df.isnull().sum()
    cobertura  = (1 - df.isnull().mean()) * 100
    resumen_cols = pd.DataFrame({
        'tipo_pandas': df.dtypes,
        'n_presentes': df.count(),
        'n_ausentes':  ausentes,
        'cobertura_%': cobertura.round(1),
        'estado':      cobertura.apply(
            lambda x: '✅ OK' if x >= umbral_variable else '❌ BAJA'
        )
    }).sort_values('cobertura_%')

    # Cobertura por registro
    cob_fila = (1 - df.isnull().mean(axis=1)) * 100
    n_filas_bajas = (cob_fila < umbral_registro).sum()

    # Cobertura global
    cob_global = df.count().sum() / n_total * 100

    # Imprimir resumen
    print(f'{'═'*55}')
    print(f'  INFORME DE COBERTURA')
    print(f'{'═'*55}')
    print(f'  Dataset:            {n_filas:,} filas × {n_cols} columnas')
    print(f'  Cobertura global:   {cob_global:.1f}%')
    print(f'  Variables < {umbral_variable:.0f}%:    {(cobertura < umbral_variable).sum()}')
    print(f'  Registros < {umbral_registro:.0f}%:   {n_filas_bajas} ({n_filas_bajas/n_filas*100:.1f}%)')
    print(f'{'─'*55}')
    print('  Cobertura por variable:')
    print(resumen_cols[['n_presentes', 'n_ausentes', 'cobertura_%', 'estado']].to_string())
    print(f'{'═'*55}')

    return resumen_cols

# Aplicamos al Titanic
resumen = informe_cobertura(titanic, umbral_variable=80.0, umbral_registro=50.0)

---
### 🔧 Ejercicio 2 — Evaluación de cobertura en un dataset propio

Crea un dataset sintético con cobertura variable y aplica el informe de cobertura.

In [ ]:
np.random.seed(99)
n = 500

# Dataset con distintos niveles de completitud por columna
df_cobertura = pd.DataFrame({
    'id':         range(n),
    'edad':       np.where(np.random.rand(n) < 0.05, np.nan, np.random.randint(18, 70, n)),  # 95%
    'ingresos':   np.where(np.random.rand(n) < 0.25, np.nan, np.random.normal(30000, 8000, n).round(0)),  # 75%
    'educacion':  np.where(np.random.rand(n) < 0.40, np.nan, np.random.choice(['Primaria','Secundaria','Universidad'], n)),  # 60%
    'cod_postal': np.where(np.random.rand(n) < 0.60, np.nan, np.random.randint(10000, 52999, n)),  # 40%
    'notas':      np.where(np.random.rand(n) < 0.80, np.nan, np.random.choice(['bueno','regular','malo'], n)),  # 20%
})

# TAREA 1: Aplica la función informe_cobertura() con umbral_variable=70% y umbral_registro=60%
# informe_cobertura(df_cobertura, umbral_variable=70.0, umbral_registro=60.0)

# TAREA 2: ¿Qué variables descartarías antes de entrenar un modelo? ¿Por qué?
# Respuesta: ...

# TAREA 3: Visualiza la cobertura por variable con un gráfico de barras horizontal
# ...

# TAREA 4: Identifica los registros con menos del 60% de completitud
#           ¿Cuántos son? ¿Los eliminarías o los imputarías?
# ...

print('Completa las tareas comentadas arriba.')

---
### 🔧 Ejercicio 3 — Informe de calidad completo del Bloque 3

Integra todo lo aprendido en el Bloque 3 en un único pipeline de diagnóstico de calidad aplicado a un dataset a tu elección.

In [ ]:
# Elige un dataset: tips, diamonds, penguins, o uno externo (Kaggle, UCI)
# df_eval = sns.load_dataset('tips')

# ETAPA 1 — INSPECCIÓN INICIAL
#   shape, dtypes, head, info, describe

# ETAPA 2 — VALORES AUSENTES (contenido 3.1)
#   isnull().sum(), missingno.matrix, diagnóstico MCAR/MAR/MNAR

# ETAPA 3 — OUTLIERS (contenido 3.2)
#   detectar_outliers_iqr para cada variable numérica
#   boxplots visuales
#   ¿son errores o valores legítimos?

# ETAPA 4 — DATOS INCORRECTOS (contenido 3.3)
#   duplicados: duplicated().sum()
#   valores imposibles: revisar rangos válidos
#   normalización de texto: str.strip(), str.title()

# ETAPA 5 — CODIFICACIÓN (contenido 3.4)
#   revisar tipos: ¿hay numéricas como object? ¿categóricas sin orden definido?
#   corregir con astype(), CategoricalDtype, pd.to_numeric, pd.to_datetime

# ETAPA 6 — COBERTURA (contenido 3.5)
#   informe_cobertura(df_eval)
#   ¿qué variables o registros descartarías?

# CONCLUSIONES
#   Documenta los problemas encontrados y las decisiones tomadas.

print('Completa el pipeline de diagnóstico de calidad.')

---
## 📝 Resumen de la sesión

| Concepto | Clave |
|----------|-------|
| Duplicados | `df.duplicated()` / `df.drop_duplicates()`. Siempre revisar por columna clave, no solo por fila completa. |
| Errores tipográficos | Normalizar con `.str.strip().str.lower().str.title()` y mapeo manual de variantes. |
| Valores imposibles | Definir reglas de dominio (min/max) y marcar como NaN los que las incumplan. |
| Formatos heterogéneos | Teléfonos: `re.sub` + validar longitud. Fechas: `pd.to_datetime(errors='coerce')`. |
| Encoding de archivo | `pd.read_csv(encoding='latin-1')`. Detectar con `chardet`. |
| Mezcla de tipos | `pd.to_numeric(errors='coerce')` convierte lo no convertible en NaN. |
| Categórica ordinal | `CategoricalDtype(categories=[...], ordered=True)` para definir el orden. |
| Cobertura por variable | `(1 - df.isnull().mean()) * 100`. Umbral típico de aceptabilidad: 80%. |
| Cobertura por registro | `(1 - df.isnull().mean(axis=1)) * 100`. Registros muy incompletos → candidatos a descartar. |

### 🔜 Próximo bloque
**Bloque 4 — Sesión 1:** Metadatos y diccionario de datos. Identificación de entidades, relaciones y volumetrías.